# Frozen-Rule Evaluation (`pipelines/05_rule_evaluation.ipynb`)

Evaluate the **frozen** monitoring rules (`ruleset-v1.0-frozen`) on the holdout evaluation period (**January 29–February 11, 2019**; `14` daily batches, `24,059` expected transactions) by comparing `workspace.sentinel_gold.alerts` against the separate fraud labels in `/Volumes/workspace/sentinel_evaluation/evaluation_labels/replay_v1`.

- **No label leakage**: Fraud labels (`is_fraud`) remain outside Bronze, Silver, historical features, and `workspace.sentinel_gold.alerts`.
- **Frozen thresholds**: Thresholds and rule versions (`ruleset-v1.0-frozen`) were frozen before releasing evaluation batches and must not be tuned using evaluation outcomes.

In [ ]:
# Step 1: Configuration and inputs for the January 29 - February 11, 2019 evaluation period
from pyspark.sql import functions as F

EVALUATION_START_DATE = "2019-01-29"
EVALUATION_END_DATE = "2019-02-11"
FROZEN_RULESET_VERSION = "ruleset-v1.0-frozen"

SILVER_TRANSACTIONS_TABLE = "workspace.sentinel_silver.transactions"
GOLD_ALERTS_TABLE = "workspace.sentinel_gold.alerts"
MANIFEST_PATH = "/Volumes/workspace/sentinel_sources/replay_staging/replay_v1/manifest"
EVALUATION_LABELS_PATH = "/Volumes/workspace/sentinel_evaluation/evaluation_labels/replay_v1"

### Step 2: Load Evaluation-Period Transactions, Alerts, and Separate Labels

We filter accepted Silver transactions, Gold alerts, and separate evaluation labels to `replay_batch` between `2019-01-29` and `2019-02-11`, verify that all `14` evaluation batches match the manifest, and confirm that `is_fraud` never entered Silver or Gold.

In [ ]:
# Step 2: Read Silver transactions, Gold alerts, and separate fraud labels for Jan 29 - Feb 11
silver_df = spark.read.table(SILVER_TRANSACTIONS_TABLE)
gold_alerts_df = spark.read.table(GOLD_ALERTS_TABLE)
manifest_df = spark.read.parquet(MANIFEST_PATH)
labels_df = spark.read.parquet(EVALUATION_LABELS_PATH)

# Confirm fraud labels remain excluded from Silver and Gold.
assert "is_fraud" not in silver_df.columns, "Fraud labels must not appear in Silver"
assert "is_fraud" not in gold_alerts_df.columns, "Fraud labels must not appear in Gold alerts"

# Expected evaluation count from the manifest.
eval_manifest_df = manifest_df.filter(
    F.col("replay_batch").between(F.lit(EVALUATION_START_DATE), F.lit(EVALUATION_END_DATE))
)
expected_eval_rows = (
    eval_manifest_df.agg(F.sum("expected_transaction_count")).collect()[0][0]
)

eval_silver_df = silver_df.filter(
    F.col("replay_batch").between(F.lit(EVALUATION_START_DATE), F.lit(EVALUATION_END_DATE))
)
eval_alerts_df = gold_alerts_df.filter(
    F.col("replay_batch").between(F.lit(EVALUATION_START_DATE), F.lit(EVALUATION_END_DATE))
)
eval_labels_df = (
    labels_df.filter(
        F.col("replay_batch").between(F.lit(EVALUATION_START_DATE), F.lit(EVALUATION_END_DATE))
    )
    .select(
        "trans_num",
        F.col("is_fraud").cast("int").alias("actual_fraud"),
    )
)

actual_eval_rows = eval_silver_df.count()
assert actual_eval_rows == expected_eval_rows, (
    f"Expected {expected_eval_rows:,} evaluation transactions in Silver "
    f"(release and ingest Jan 29 - Feb 11 first); found {actual_eval_rows:,}"
)

# Verify all evaluation alerts use the frozen ruleset version.
non_frozen_alerts = eval_alerts_df.filter(
    F.col("ruleset_version") != F.lit(FROZEN_RULESET_VERSION)
).count()
assert non_frozen_alerts == 0, (
    f"Expected all evaluation alerts to use {FROZEN_RULESET_VERSION}"
)

print(f"Verified {actual_eval_rows:,} evaluation transactions ({EVALUATION_START_DATE} to {EVALUATION_END_DATE}).")

### Step 3: Compute Confusion Matrix (`TP`, `FP`, `FN`, `TN`), Precision, and Recall

We join the evaluation-period Silver transactions (`1` row per `trans_num`) with the separate fraud labels (`actual_fraud` in `{0, 1}`) and the frozen Gold alerts (`is_flagged`), then calculate:
- **TP (True Positives)**: `is_flagged == True` and `actual_fraud == 1`
- **FP (False Positives)**: `is_flagged == True` and `actual_fraud == 0`
- **FN (False Negatives)**: `is_flagged == False` and `actual_fraud == 1`
- **TN (True Negatives)**: `is_flagged == False` and `actual_fraud == 0`
- **Precision**: `TP / (TP + FP)` (returns `0.0` when `TP + FP == 0`)
- **Recall**: `TP / (TP + FN)` (returns `0.0` when `TP + FN == 0`)

In [ ]:
# Step 3: Join evaluation transactions with separate labels and frozen Gold alerts (in-memory only)
eval_comparison_df = (
    eval_silver_df.select("trans_num", "replay_batch")
    .join(eval_labels_df, on="trans_num", how="inner")
    .join(
        eval_alerts_df.select(
            "trans_num",
            F.lit(True).alias("is_flagged"),
            "matched_rule_ids",
        ),
        on="trans_num",
        how="left",
    )
    .withColumn("is_flagged", F.coalesce(F.col("is_flagged"), F.lit(False)))
)

assert eval_comparison_df.count() == expected_eval_rows, (
    "Evaluation join row count does not match expected evaluation transactions"
)

counts_df = eval_comparison_df.agg(
    F.count("*").alias("evaluated_transactions"),
    F.sum(F.when(F.col("is_flagged"), 1).otherwise(0)).alias("flagged_alerts"),
    F.sum(F.when(F.col("actual_fraud") == 1, 1).otherwise(0)).alias("actual_fraud_transactions"),
    F.sum(F.when(F.col("is_flagged") & (F.col("actual_fraud") == 1), 1).otherwise(0)).alias("TP"),
    F.sum(F.when(F.col("is_flagged") & (F.col("actual_fraud") == 0), 1).otherwise(0)).alias("FP"),
    F.sum(F.when((~F.col("is_flagged")) & (F.col("actual_fraud") == 1), 1).otherwise(0)).alias("FN"),
    F.sum(F.when((~F.col("is_flagged")) & (F.col("actual_fraud") == 0), 1).otherwise(0)).alias("TN"),
)

# Compute precision and recall safely, handling zero denominators.
evaluation_metrics_df = (
    counts_df
    .withColumn(
        "precision",
        F.when(
            (F.col("TP") + F.col("FP")) > 0,
            F.round(F.col("TP") / (F.col("TP") + F.col("FP")), 4),
        ).otherwise(F.lit(0.0)),
    )
    .withColumn(
        "recall",
        F.when(
            (F.col("TP") + F.col("FN")) > 0,
            F.round(F.col("TP") / (F.col("TP") + F.col("FN")), 4),
        ).otherwise(F.lit(0.0)),
    )
)

metrics_row = evaluation_metrics_df.collect()[0]
assert (
    metrics_row["TP"] + metrics_row["FP"] + metrics_row["FN"] + metrics_row["TN"]
    == metrics_row["evaluated_transactions"]
), "TP + FP + FN + TN must equal evaluated_transactions"

display(evaluation_metrics_df)